# Analisis jaringan: sentralitas, k-core, komponen
        **graph › analysis** · notebook `network_analysis`

        **Kapan dipakai:** Langkah pertama pada data graf baru: siapa simpul penting, seberapa terhubung, apakah ada hub dan komponen terpisah.

        **Jebakan umum:** Sentralitas berbeda mengukur hal berbeda (derajat ≠ betweenness ≠ PageRank). Hitung pada graf tak berarah/terarah sesuai makna data.

**Data demo:** Cora (2708 makalah, 5429 sitasi, 1433 fitur kata, 7 topik).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Format data: `nodes.csv` (id, label, fitur...) dan `edges.csv` (source, target; arah diabaikan).

In [ ]:
GRAPH_DIR = "data_latihan/graph/cora"
MODEL_KEY = "network_analysis"
SEED = 42
OUT = out_dir("graph", "analysis", MODEL_KEY); set_seed(SEED)

## 2. Muat graf dan ringkasan

In [ ]:
from superdb.graph import load_graph
g = load_graph(GRAPH_DIR)
deg = np.asarray(g.A.sum(1)).ravel(); E = int(g.A.nnz // 2)
same = float((g.y[g.A.nonzero()[0]] == g.y[g.A.nonzero()[1]]).mean())
print(f"{g.n} simpul, {E} sisi, derajat rata-rata {deg.mean():.2f} (maks {int(deg.max())}), simpul terisolasi: {int((deg == 0).sum())}, fitur: {None if g.X is None else g.X.shape[1]}")
print(f"homofili sisi (porsi sisi yang menghubungkan simpul sekelas): {same:.2f}  -> {'graf homofili: tetangga informatif' if same > .6 else 'graf heterofili: tetangga kurang informatif'}")

## 3. Distribusi derajat
Jaringan nyata sering skala-bebas: sedikit hub dengan derajat sangat besar. Garis lurus di plot log-log menandakan ekor berat.

In [ ]:
import networkx as nx
G = nx.from_scipy_sparse_array(g.A)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2)); ax[0].hist(deg, bins=40); ax[0].set(title="distribusi derajat", xlabel="derajat")
vals, cnt = np.unique(deg[deg > 0], return_counts=True); ax[1].loglog(vals, cnt / cnt.sum(), "o"); ax[1].set(title="log-log", xlabel="derajat", ylabel="proporsi")
plt.tight_layout(); plt.savefig(OUT / "degree.png", dpi=130); plt.show()

## 4. Komponen terhubung, klaster, assortativity

In [ ]:
comps = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
stats = {"n_komponen": len(comps), "komponen_terbesar": comps[0], "porsi_komponen_terbesar": comps[0] / g.n, "koefisien_klaster_rata2": nx.average_clustering(G),
         "transitivity": nx.transitivity(G), "assortativity_derajat": nx.degree_assortativity_coefficient(G), "kerapatan": nx.density(G)}
display(show_table(stats))

## 5. Simpul terpenting menurut beberapa ukuran
Ukuran berbeda menjawab pertanyaan berbeda: **derajat** (populer), **PageRank** (dirujuk simpul penting), **betweenness** (jembatan antar-komunitas), **k-core** (inti padat).

In [ ]:
pr = nx.pagerank(G); bt = nx.betweenness_centrality(G, k=300, seed=0); core = nx.core_number(G)
T = pd.DataFrame({"derajat": deg, "pagerank": pd.Series(pr), "betweenness": pd.Series(bt), "k_core": pd.Series(core), "kelas": [g.classes[i] for i in g.y]})
display(T.sort_values("pagerank", ascending=False).head(8).round(4))
print("korelasi peringkat (Spearman) antar ukuran:"); display(T[["derajat", "pagerank", "betweenness", "k_core"]].corr(method="spearman").round(2))
print("distribusi k-core:", dict(pd.Series(core).value_counts().sort_index()))
T.round(5).to_csv(OUT / "centralities.csv"); save_json({"stats": stats}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Fitur-fitur ini (derajat, PageRank, k-core, ...) bisa langsung menjadi fitur simpul untuk model (lihat `node_classification/structure_lgbm`).
- Untuk graf berarah, hitung ukuran in/out-degree terpisah dan PageRank terarah.